Updated inference defaults: local labels use empirical adjusted p-values; zero permutations are untested. ICG uses within-type contact flags and an explicit expression scale. Recorded outputs were cleared and require rerunning. See `docs/analysis.md` for assumptions.


# Parameter Deep Dive

Most tutorials pick one value for each parameter and move on. This notebook does the opposite: for each parameter, it asks *why that value, and what happens if I change it?*

Each section sweeps one parameter across a realistic range and shows what actually changes — not just in theory, but in the actual numbers. The goal: know when a default is safe to trust, and when your data needs something different.

Dataset: `ab.datasets.xenium()` (~162k cells, 377 genes), the same one used throughout this notebook series.


In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astrolabe as ab
import scanpy as sc


## 1. Setup

The usual load → filter → normalize steps from `01_core_workflow.ipynb`. Every section below builds on this. If a section needs to try something without disturbing later sections, it saves its result under its own `key_added` name instead of overwriting the shared default.


In [ ]:
mdata = ab.datasets.xenium()
ab.pipeline.filter_xenium(mdata)
ab.pp.normalize_total(mdata)
ab.pp.log1p(mdata)
ab.pp.highly_variable_genes(mdata, n_top_genes=2000)

ab.summary(mdata)


## 2. Spatial graph and clusters for this section

This section revisits the spatial-statistics tests from `02_spatial_statistics.ipynb` — Moran's I, Geary's C, LISA, Getis-Ord, bivariate Moran's I, Ripley's L, and the genome-wide SVG scan. This time we're not changing *what* we test, we're changing *how* the test is run: how many permutations, how significance is approximated, which spatial graph is used.

To keep the numbers comparable to `02`, we rebuild the same setup: PCA + clustering (`resolution=0.5`) and a Delaunay spatial graph. `EPCAM` (tumor marker) and `PTPRC` (immune marker) are the same two genes used throughout this series.


In [ ]:
ab.tl.pca(mdata)
ab.tl.transcriptional_neighbors(mdata)
ab.tl.cluster(mdata, resolution=0.5, neighbors_key='neighbors')

# Spatial graph the rest of this notebook needs -- coexists independently
# of the transcriptomic graph built above.
ab.tl.spatial_neighbors(mdata, coord_type='delaunay', delaunay_max_dist=200)

gene = 'EPCAM'
gene2 = 'PTPRC'
assert gene in mdata['xenium'].var_names and gene2 in mdata['xenium'].var_names

mdata['xenium'].obs['leiden'].value_counts()


## 3. `n_perms`: how many permutations does a test actually need?

Every one of these tests asks: is this pattern real, or could it happen by chance? `n_perms` controls how that "by chance" baseline gets estimated. But not every function has a shortcut when `n_perms=0`:

- **`moran`, `geary_c`, `getis_ord`** can skip permutations — they fall back to a fast, analytic approximation instead. Here, `n_perms=0` vs. a large `n_perms` is a real comparison: does the fast shortcut agree with the slow, simulation-based answer?
- **`moran_local`, `moran_local_bv`** have no shortcut. At `n_perms=0` they simply return no significance test at all (`NaN`).
- **`moran_bv`** refuses to run without permutations — it raises an error.
- **`ripley_l`** is the same story as the local Moran functions: no permutations, no result.

So this section asks two different questions. **3.1–3.3**: does the fast shortcut match the slow answer? **3.4–3.7**: for the tests with no shortcut, how many permutations do you actually need before the answer stops changing?


### 3.1 `tl.moran`: fast shortcut vs. permutations

`02` reported Moran's I of 0.478 (EPCAM) and 0.258 (PTPRC). `I`, `EI`, `z_norm`, and `p_norm` come from a formula — they never depend on `n_perms`. Only `z_sim`/`p_sim` come from permutations.


In [ ]:
t0 = time.perf_counter()
moran_analytic = ab.tl.moran(mdata, [gene, gene2], n_perms=0, key_added='moran_analytic')
t_analytic = time.perf_counter() - t0

t0 = time.perf_counter()
moran_perm = ab.tl.moran(mdata, [gene, gene2], n_perms=1000, key_added='moran_perm')
t_perm = time.perf_counter() - t0

print(f'n_perms=0    ({t_analytic:.2f}s):')
print(moran_analytic)
print(f'\nn_perms=1000 ({t_perm:.2f}s):')
print(moran_perm)
print(f'\nSpeedup: {t_perm / t_analytic:.1f}x')


Same numbers either way: `I`, `EI`, `z_norm`, `p_norm` are identical whether `n_perms=0` or `1000`. The only difference is that `n_perms=1000` also gives you `z_sim`/`p_sim` — and both paths agree EPCAM and PTPRC are highly significant. For one gene pair the extra time barely matters, but this is exactly why `spatially_variable_genes` defaults to `n_perms=0` (Section 5): permutations cost time per gene, and scanning hundreds of genes adds up fast.


### 3.2 `tl.geary_c`: same story as Moran's I

Same setup as 3.1: `C`, `EC`, `z_norm`, `p_norm` come from a formula and don't move with `n_perms`; `z_sim`/`p_sim` only show up once `n_perms>0`.


In [ ]:
t0 = time.perf_counter()
geary_analytic = ab.tl.geary_c(mdata, [gene, gene2], n_perms=0, key_added='geary_c_analytic')
t_analytic = time.perf_counter() - t0

t0 = time.perf_counter()
geary_perm = ab.tl.geary_c(mdata, [gene, gene2], n_perms=1000, key_added='geary_c_perm')
t_perm = time.perf_counter() - t0

print(f'n_perms=0    ({t_analytic:.2f}s):')
print(geary_analytic)
print(f'\nn_perms=1000 ({t_perm:.2f}s):')
print(geary_perm)


Same result as Moran's I: C ≈ 0.520 (EPCAM), 0.739 (PTPRC), unchanged either way — matching `02`. Both statistics agree EPCAM is more spatially organized than PTPRC (higher I, lower C), just measured two different ways.


### 3.3 `tl.getis_ord`: two different nulls, one map convention

`getis_ord` retains `z_norm`/`p_norm` as analytic diagnostics. With
`n_perms=0`, empirical significance and `p_adj` are NaN and labels are
`not_tested`. With permutations, `p_sim` uses empirical two-sided equal tails
and `p_adj` corrects all requested cells and fields together. Hot/cold labels
use `p_adj < alpha`. The comparison below contrasts untested analytic
diagnostics with empirical significance.

In [ ]:
go_analytic = ab.tl.getis_ord(mdata, [gene], n_perms=0, alpha=0.05, n_jobs=4, key_added='getis_ord_analytic')
go_perm = ab.tl.getis_ord(mdata, [gene], n_perms=1000, alpha=0.05, n_jobs=4, key_added='getis_ord_perm')

print('n_perms=0 (analytic normal null; map gate = raw two-sided p_norm) counts:')
print(go_analytic.loc[gene, 'type'].value_counts())
print('\nn_perms=1000 (permutation null; map gate = adjusted empirical p_adj) counts:')
print(go_perm.loc[gene, 'type'].value_counts())

The zero-permutation path is explicitly not tested. It cannot be compared
as a significance map with the empirical permutation path. Analytic normal
diagnostics can be miscalibrated for sparse count tails; use empirical p-values
for reported inference and enough permutations for the required FDR resolution.


### 3.4–3.7: no shortcut — so how many permutations is enough?

`moran_local`, `moran_local_bv`, `moran_bv`, and `ripley_l` have no fast path: at `n_perms=0` you either get `NaN` or an error. So instead of "0 vs. 1000," the real question here is: how many permutations does it take before the answer stops changing?


### 3.4 `tl.moran_local`: how many permutations does it need?

With `n_perms=0`, `p_sim`, `p_dir`, `z_sim`, and `p_adj` are NaN.
`quadrant` is `not_tested`; `quadrant_raw` contains exploratory sign labels.
The statistic and raw quadrant are fixed by the data; significance requires
permutations and is gated on adjusted empirical p-values.

In [ ]:
zero_check = ab.tl.moran_local(mdata, [gene], n_perms=0, key_added='moran_local_zero')
print('n_perms=0 p_sim all NaN:', zero_check.loc[gene, 'p_sim'].isna().all())

perm_counts = [100, 1000]
lisa_runs = {}
timings = {}
for n in perm_counts:
    t0 = time.perf_counter()
    lisa_runs[n] = ab.tl.moran_local(mdata, [gene], n_perms=n, n_jobs=4, key_added=f'moran_local_n{n}')
    timings[n] = time.perf_counter() - t0

for n in perm_counts:
    res = lisa_runs[n].loc[gene]
    n_map = (res['quadrant'] != 'NS').sum()
    n_fdr = (res['p_adj'] < 0.05).sum()
    print(f'n_perms={n:>5} ({timings[n]:.1f}s): adjusted-map (p_adj<0.05) = {n_map:,}; stringent (p_adj<0.05) = {n_fdr:,}')

ii_match = all(np.allclose(lisa_runs[100].loc[gene, 'Ii'], lisa_runs[n].loc[gene, 'Ii']) for n in perm_counts)
print(f'\nIi identical across all n_perms settings: {ii_match}')

`Ii` stays fixed with `n_perms`. Empirical p-values and their BH corrections
change with permutation resolution. The equal-tail floor is
`2/(n_perms+1)`; large cell-by-field families may require substantially more
permutations. Coarse resolution can limit power. Labels use the same `p_adj`
threshold as the inferential table; directed smaller tails remain diagnostics.


### 3.5 `tl.moran_local_bv`: same treatment, two genes at once

Same as `moran_local`: no shortcut, `NaN` significance at `n_perms=0` (raw ungated quadrants with a warning). One expectation to set before running it: the per-cell *bivariate* test randomizes only each cell's neighborhood values of the second gene, and on a sparse spatial graph that null is wide — so far fewer cells clear the gate than in the univariate LISA, even when the global bivariate signal is strong.

In [ ]:
pair_label = f'{gene}:{gene2}'
bv_local_runs = {}
bv_timings = {}
for n in perm_counts:
    t0 = time.perf_counter()
    bv_local_runs[n] = ab.tl.moran_local_bv(mdata, gene, gene2, n_perms=n, n_jobs=4, key_added=f'moran_local_bv_n{n}')
    bv_timings[n] = time.perf_counter() - t0

for n in perm_counts:
    res = bv_local_runs[n].loc[pair_label]
    n_map = (res['quadrant'] != 'NS').sum()
    n_fdr = (res['p_adj'] < 0.05).sum()
    print(f'n_perms={n:>5} ({bv_timings[n]:.1f}s): adjusted-map (p_adj<0.05) = {n_map:,}; stringent (p_adj<0.05) = {n_fdr:,}')

ii_match = all(
    np.allclose(bv_local_runs[100].loc[pair_label, 'Ii'], bv_local_runs[n].loc[pair_label, 'Ii'])
    for n in perm_counts
)
print(f'\nIi identical across all n_perms settings: {ii_match}')

Same conclusion as 3.4: `Ii` is fixed, only the significance counts shift with `n_perms`. Notice how much smaller the gated counts are than in the univariate case — that is the low per-cell power of the bivariate local test discussed above, not an absence of structure (the global bivariate `I` between these two genes is strongly negative). Read the bivariate LISA map as "where is the cross-association most coherent", and lean on the global `moran_bv` for the existence question.

### 3.6 `tl.moran_bv`: no shortcut, no exceptions

`moran_bv` won't even try without permutations — it raises an error at `n_perms=0` instead of returning `NaN`. It also has no `seed` option, unlike `ripley_l`, so its permutation results aren't exactly reproducible run to run.


In [ ]:
try:
    ab.tl.moran_bv(mdata, gene, gene2, n_perms=0)
except ValueError as e:
    print(f'n_perms=0 raised ValueError: {e}')

bv_runs = {}
bv_g_timings = {}
for n in perm_counts:
    t0 = time.perf_counter()
    bv_runs[n] = ab.tl.moran_bv(mdata, gene, gene2, n_perms=n, key_added=f'moran_bv_n{n}')
    bv_g_timings[n] = time.perf_counter() - t0

comparison = pd.concat(
    {n: bv_runs[n].loc[(gene, gene2)] for n in perm_counts}, axis=1
).T
comparison.index.name = 'n_perms'
print(comparison)


`I` is fixed by the data, but here `EI` is *also* estimated from permutations rather than a formula, so it drifts slightly with `n_perms` and between runs. The smallest p-value you can get is `1/(n_perms+1)`: about 0.0099 at 100 permutations, 0.001 at 1000. For a signal this strong, both are already far below any significance threshold — more permutations wouldn't change the conclusion, but they would matter for a borderline result.


### 3.7 `tl.ripley_l`: how many permutations for a stable envelope?

`ripley_l` builds its own random baseline from scratch — no `esda` involved. At `n_perms=0` there's no baseline at all: `L_hi`, `L_lo`, and `p_value` are all `NaN`. As in `02`, we subsample per cluster to keep this fast — 2,000 cells per cluster instead of `02`'s 10,000, since this section reruns the test many times.


In [ ]:
rng = np.random.default_rng(42)
max_per_group = 2_000

xen = mdata['xenium']
mask = np.zeros(xen.n_obs, dtype=bool)
for g in xen.obs['leiden'].cat.categories:
    idx = np.where(xen.obs['leiden'] == g)[0]
    chosen = rng.choice(idx, min(len(idx), max_per_group), replace=False)
    mask[chosen] = True
xen_sub = xen[mask].copy()
print(f"Subsampled to {xen_sub.n_obs} cells ({max_per_group} per group max, {xen.obs['leiden'].nunique()} groups)")

zero_env = ab.tl.ripley_l(xen_sub, groupby='leiden', n_perms=0, key_added='ripley_l_zero')
biggest_cluster = xen_sub.obs['leiden'].value_counts().idxmax()
print(f"n_perms=0, cluster '{biggest_cluster}': L_hi all NaN = {zero_env[biggest_cluster]['L_hi'].isna().all()}")

ripley_runs = {}
ripley_timings = {}
for n in perm_counts:
    t0 = time.perf_counter()
    ripley_runs[n] = ab.tl.ripley_l(xen_sub, groupby='leiden', n_perms=n, seed=42, n_jobs=4, key_added=f'ripley_l_n{n}')
    ripley_timings[n] = time.perf_counter() - t0

for n in perm_counts:
    curve = ripley_runs[n][biggest_cluster]
    envelope_width = (curve['L_hi'] - curve['L_lo']).mean()
    p_at_max = curve['p_value'].iloc[-1]
    print(f"n_perms={n:>5} ({ripley_timings[n]:.1f}s): mean envelope width = {envelope_width:.2f}, p_value at max r = {p_at_max:.4f}")

(ab.pl.ripley_plot(xen_sub, key='ripley_l_n100', cell_types=[biggest_cluster], title='n_perms=100')
 | ab.pl.ripley_plot(xen_sub, key='ripley_l_n1000', cell_types=[biggest_cluster], title='n_perms=1000')).show()


As expected, no baseline at `n_perms=0`. Above that, the baseline barely moves: mean width 6.79 µm at 100 permutations vs. 6.96 µm at 1000 — a ~2.5% difference, basically noise. `L_obs` (the actual measured clustering) never changes, since it doesn't depend on permutations at all. And the significance result (`p_value = 0.0000`) is identical either way — for a signal this strong, 100 permutations already tells you everything 1000 would.


**Summary.** `moran`, `geary_c`, and `getis_ord` all have a fast shortcut — but only two of the three (`moran`, `geary_c`) can be trusted at `n_perms=0`. `getis_ord`'s shortcut missed an entire population of cold spots that the slow path found. The other four functions have no shortcut at all, and permutation count matters more than you'd expect: even for a strong, obvious signal, 100 permutations wasn't quite enough for stable per-cell results in 3.4. The general rule: trust the fast path for whole-tissue statistics, be more careful with per-cell ones, and don't assume "the signal is strong" means "100 permutations is plenty."


## 4. Other hyperparameters that matter

`n_perms` isn't the only knob. This section covers the multiple-testing correction method, the hot/cold significance threshold, Ripley's L resolution (and a hidden distance limit), and the biggest lever of all: which spatial graph the test runs against.


### 4.1 `p_adjust_method`: does the correction method matter?

`moran_local`, `getis_ord`, and `moran_local_bv` correct empirical
two-sided `p_sim` across all requested cells and fields in one call.
BH is the default and assumes independence or positive dependence;
`p_adjust_method='fdr_by'` accommodates arbitrary dependence. Bonferroni
controls family-wise error. Finite permutations bound attainable p-values;
normal approximations are not substituted to bypass that limit.

In [ ]:
methods = ['fdr_bh', 'bonferroni']

for method in methods:
    lisa = ab.tl.moran_local(mdata, [gene], n_perms=1000, p_adjust_method=method, n_jobs=4, key_added=f'moran_local_{method}')
    n_sig = (lisa.loc[gene, 'p_adj'] < 0.05).sum()
    print(f"moran_local      p_adjust_method='{method:<10}': {n_sig:>7,} / {mdata['xenium'].n_obs:,} cells significant (p_adj)")

for method in methods:
    go = ab.tl.getis_ord(mdata, [gene], n_perms=1000, p_adjust_method=method, n_jobs=4, key_added=f'getis_ord_{method}')
    n_sig = (go.loc[gene, 'p_adj'] < 0.05).sum()
    print(f"getis_ord        p_adjust_method='{method:<10}': {n_sig:>7,} / {mdata['xenium'].n_obs:,} cells significant (p_adj)")

for method in methods:
    bv_local = ab.tl.moran_local_bv(mdata, gene, gene2, n_perms=1000, p_adjust_method=method, n_jobs=4, key_added=f'moran_local_bv_{method}')
    n_sig = (bv_local.loc[pair_label, 'p_adj'] < 0.05).sum()
    print(f"moran_local_bv   p_adjust_method='{method:<10}': {n_sig:>7,} / {mdata['xenium'].n_obs:,} cells significant (p_adj)")

Different corrections have different error targets and power.
Bonferroni controls family-wise error; BH controls FDR under its assumptions,
and BY addresses arbitrary dependence. The choice also controls the map:
quadrant/hotspot labels use adjusted empirical p-values. Interpret low power
in light of family size and permutation resolution.


### 4.2 `alpha`: the hot/cold threshold

`alpha` gates labels on `p_adj`, retaining raw statistics and p-values.
It therefore changes the requested significance threshold, while the
permutation null and correction family stay fixed. The reclassification
below uses the already computed adjusted empirical p-values.

In [ ]:
# Reclassify the n_perms=1000 getis_ord result (go_perm, from 3.3) at different
# alpha thresholds directly, matching the function's own hot/cold/NS logic
# (map gate = directed pseudo-p) -- no need to rerun the permutation test
# Reuse the adjusted empirical p-values at each alpha.
z_vals = go_perm.loc[gene, 'z_sim'].to_numpy()
p_adj_vals = go_perm.loc[gene, 'p_adj'].to_numpy()

for alpha in [0.01, 0.05, 0.10]:
    hot = ((z_vals > 0) & (p_adj_vals < alpha)).sum()
    cold = ((z_vals < 0) & (p_adj_vals < alpha)).sum()
    print(f'alpha={alpha:.2f}: hot={hot:>7,}  cold={cold:>7,}  NS={mdata["xenium"].n_obs - hot - cold:>7,}')

As expected: hot/cold counts grow as `alpha` gets looser (0.01 → 0.05 → 0.10). Unlike `n_perms` or `p_adjust_method`, `alpha` doesn't change the test itself — it just decides where to draw the line afterward.


### 4.3 `ripley_l`: resolution (`support`) and a hidden distance limit

`support` is just how many distance steps get plotted — more steps, smoother curve, longer runtime. The maximum distance tested (`max_r`) isn't a parameter at all: it's calculated automatically from the tissue's size, and there's no way to change it directly.


In [ ]:
# The internal max_r for this dataset, reproduced from ripley_l's own formula.
coords_sub = xen_sub.obsm['spatial']
xy_range = coords_sub.max(axis=0) - coords_sub.min(axis=0)
max_r_computed = float(min(xy_range)) / 4.0
print(f'Internally-computed max_r for this subsample: {max_r_computed:.1f} um (not user-settable)')

support_values = [50, 200, 500]
support_timings = {}
support_runs = {}
for s in support_values:
    t0 = time.perf_counter()
    support_runs[s] = ab.tl.ripley_l(xen_sub, groupby='leiden', support=s, n_perms=200, seed=42, n_jobs=4, key_added=f'ripley_l_support{s}')
    support_timings[s] = time.perf_counter() - t0
    n_steps = len(support_runs[s][biggest_cluster])
    print(f'support={s:>4} ({support_timings[s]:.1f}s): {n_steps} distance steps, spacing={max_r_computed / (s - 1):.2f} um')


`max_r` is fixed by how big the tissue is — not something you can tune. Runtime scales with `support`, but the shape of the curve doesn't change; `support=50` just looks blockier than `support=500`. Practical tip: use a low `support` (50–100) while exploring, and only bump it up for a final figure.


### 4.4 The spatial graph itself: the biggest lever of all

Every function here runs against whichever spatial graph you point it at — and none of them let you change how neighbors are weighted (it's always row-standardized). So the graph itself — sparse vs. dense, KNN vs. Delaunay — is really the main thing you can tune. Below, two extra KNN graphs are built alongside the Delaunay default, and `geary_c` is rerun on all three.


In [ ]:
ab.tl.spatial_neighbors(mdata, coord_type='generic', n_neighbors=6, key_added='knn6_neighbors')
ab.tl.spatial_neighbors(mdata, coord_type='generic', n_neighbors=30, key_added='knn30_neighbors')

graph_keys = {
    'delaunay (default)': 'spatial_neighbors',
    'knn, n_neighbors=6': 'knn6_neighbors',
    'knn, n_neighbors=30': 'knn30_neighbors',
}

for label, key in graph_keys.items():
    result = ab.tl.geary_c(mdata, [gene, gene2], neighbors_key=key, n_perms=0, key_added=f'geary_c_{key}')
    print(f'{label:<22} C(EPCAM)={result.loc[gene, "C"]:.3f}  C(PTPRC)={result.loc[gene2, "C"]:.3f}')

mdata['xenium'].obs[f'{gene}_quadrant_delaunay'] = ab.tl.moran_local(
    mdata, gene, neighbors_key='spatial_neighbors', n_perms=0, key_added='moran_local_delaunay_viz'
).loc[gene, 'quadrant'].to_numpy()
mdata['xenium'].obs[f'{gene}_quadrant_knn30'] = ab.tl.moran_local(
    mdata, gene, neighbors_key='knn30_neighbors', n_perms=0, key_added='moran_local_knn30_viz'
).loc[gene, 'quadrant'].to_numpy()

(ab.pl.points(mdata, color=f'{gene}_quadrant_delaunay', title='Delaunay graph')
 | ab.pl.points(mdata, color=f'{gene}_quadrant_knn30', title='KNN, n_neighbors=30')).show()


Geary's C shifts across the three graphs, but not in a simple density-ordered way. By mean degree, Delaunay (6.0) is actually the *sparsest* of the three here, not KNN-6 (6.9) -- despite "n_neighbors=6" sounding smaller, KNN graphs are symmetrized (if A lists B as a neighbor but B doesn't list A back, the edge is kept either way), which pushes the realized mean degree above the requested `n_neighbors`. In mean-degree order for EPCAM: KNN-6 (6.9, C=0.505) < Delaunay (6.0, C=0.520) < KNN-30 (32.9, C=0.627) -- but the same ordering doesn't hold for PTPRC, where Delaunay's C is actually the *lowest* of the three, not the middle. So a single "denser graph -> less clustered" story doesn't cleanly explain the whole picture. What does hold up: a much denser graph (KNN-30 here) averages each cell over a much wider area, blurring sharp compartment boundaries and pushing C toward 1 (less clustered) -- that end of the comparison is real. The narrower gap between two graphs with similar, modest mean degree (Delaunay vs. KNN-6) is more sensitive to *how* neighbors are chosen (adaptive Delaunay triangulation with a 200 µm cutoff vs. a fixed-count KNN) than to degree alone, and isn't safe to summarize as simply "sparser = more clustered." Geary's C is always relative to whichever graph you hand it, not an absolute property of the tissue -- the big-picture map (tumor vs. stroma) looks the same across all three graphs; only the fine detail at the boundary changes.

### 4.5 One more gap: reproducibility

Only `ripley_l` lets you set a `seed`. Every other function here draws its own random permutations internally with no way to fix them — so rerunning any `n_perms>0` cell above gives the same conclusion, but not the exact same numbers.


## 5. `spatially_variable_genes` hyperparameters

This function scans every gene at once instead of testing one gene at a time. It shares the same `neighbors_key`/`p_adjust_method`/`n_jobs` options as the single-gene tests, but two things are different: it defaults to `n_perms=0` (the opposite of Section 3), and it has its own selection options (`flavor`, `n_top_genes`).


### 5.1 `flavor`: Moran's I vs. Geary's C

`flavor='moran'` ranks genes by I (higher = more organized); `flavor='geary'` ranks by C (lower = more organized) — the code flips the sort automatically. Both ask a similar question, but Geary's C is more sensitive to local structure, so the two rankings won't always agree.


In [ ]:
svg_moran = ab.tl.spatially_variable_genes(mdata, flavor='moran', n_top_genes=20, n_jobs=4, key_added='svg_moran')
svg_geary = ab.tl.spatially_variable_genes(mdata, flavor='geary', n_top_genes=20, n_jobs=4, key_added='svg_geary')

top20_moran = set(svg_moran.sort_values('score', ascending=False).head(20).index)
top20_geary = set(svg_geary.sort_values('score', ascending=True).head(20).index)

overlap = top20_moran & top20_geary
print(f'Top-20 by Moran\'s I:  {sorted(top20_moran)}')
print(f'Top-20 by Geary\'s C:  {sorted(top20_geary)}')
print(f'\nOverlap: {len(overlap)}/20 genes ({len(overlap) / 20:.0%})')


The two flavors agree on 19 of the top 20 genes. Genes both agree on are the safest calls. A gene that only shows up in one list is worth double-checking — it might reflect something Geary's C is picking up that Moran's I isn't (or vice versa), not necessarily a strong, robust signal.


### 5.2 `n_top_genes`: two different rules

`n_top_genes=None` marks a gene as spatially variable if it passes `pval_adj < 0.05` — how many genes that is depends entirely on the data. Passing a number instead always returns exactly that many genes, whether or not they're actually significant. It's easy to assume `n_top_genes=20` means "the top 20 significant genes" — it doesn't check significance at all.


In [ ]:
svg_threshold = ab.tl.spatially_variable_genes(mdata, n_top_genes=None, n_jobs=4, key_added='svg_threshold')
svg_fixed20 = ab.tl.spatially_variable_genes(mdata, n_top_genes=20, n_jobs=4, key_added='svg_fixed20')

n_by_threshold = svg_threshold['spatially_variable'].sum()
n_by_fixed = svg_fixed20['spatially_variable'].sum()
print(f"n_top_genes=None:  {n_by_threshold} genes pass pval_adj < 0.05")
print(f"n_top_genes=20:    {n_by_fixed} genes (always exactly 20, by rank)")

set_threshold = set(svg_threshold[svg_threshold['spatially_variable']].index)
set_fixed20 = set(svg_fixed20[svg_fixed20['spatially_variable']].index)
print(f"\nAll of the fixed-20 set also significant at pval_adj<0.05: {set_fixed20 <= set_threshold}")
print(f"Genes in the threshold set but outside the top 20 by rank: {len(set_threshold - set_fixed20)}")


In this dataset, 354 of 377 genes pass the significance threshold — this tissue is extremely spatially organized, so almost everything qualifies. The 20 genes picked by rank all fall inside that larger significant set here. But that's not guaranteed in general: on a weaker dataset, `n_top_genes=20` could quietly hand back genes that aren't actually significant.


### 5.3 Why `spatially_variable_genes` defaults to `n_perms=0`

Every function in Section 3 defaulted to `n_perms=1000`. This one defaults to 0, because permutation cost multiplies by the number of genes — testing hundreds of genes with permutations gets expensive fast. To show this without waiting forever, we test just the top 25 most variable genes instead of all 377.


In [ ]:
top25_genes = mdata['xenium'].var['dispersions_norm'].sort_values(ascending=False).head(25).index.tolist()
xen_svg_subset = mdata['xenium'][:, top25_genes].copy()

t0 = time.perf_counter()
svg_analytic_subset = ab.tl.spatially_variable_genes(xen_svg_subset, n_perms=0, n_jobs=-1, key_added='svg_subset_analytic')
t_analytic = time.perf_counter() - t0

t0 = time.perf_counter()
svg_perm_subset = ab.tl.spatially_variable_genes(xen_svg_subset, n_perms=1000, n_jobs=-1, key_added='svg_subset_perm')
t_perm = time.perf_counter() - t0

print(f'{len(top25_genes)} genes, n_perms=0    ({t_analytic:.2f}s)')
print(f'{len(top25_genes)} genes, n_perms=1000 ({t_perm:.2f}s), speedup from analytic: {t_perm / t_analytic:.1f}x')

comparison = svg_analytic_subset[['score', 'pval', 'pval_adj']].join(
    svg_perm_subset[['pval', 'pval_adj']], lsuffix='_analytic', rsuffix='_perm'
)
print(comparison.sort_values('score', ascending=False).head(10))


Even with only 25 genes, `n_perms=1000` clearly takes longer than `n_perms=0`. Scale that to hundreds or thousands of genes and the gap grows a lot. `score` is identical either way; for genes with a strong signal, both p-values agree it's basically zero. Any disagreement shows up in the weaker genes near the bottom of the list, where the two approaches are more likely to differ. This is why the default is `n_perms=0`: use it for scanning, and save permutations for genes you've already flagged as interesting.


### 5.4 `p_adjust_method` across the whole gene panel

Unlike Section 4.1 (150,000+ cells), this correction only runs across 377 genes — a much smaller number of tests. Bonferroni isn't automatically doomed here the way it was for cells.


In [ ]:
for method in ['fdr_bh', 'bonferroni']:
    svg = ab.tl.spatially_variable_genes(mdata, n_top_genes=None, p_adjust_method=method, n_jobs=4, key_added=f'svg_{method}')
    n_sv = svg['spatially_variable'].sum()
    print(f"p_adjust_method='{method:<10}': {n_sv} / {svg.shape[0]} genes spatially variable")


With only 377 tests, Bonferroni can still find real hits — it's just stricter than FDR-BH, not mathematically incapable like it was for cells. FDR-BH still finds at least as many genes (354 vs. 326), since it's the more permissive of the two.


### 5.5 Two more things worth knowing

- **`n_jobs`** speeds up the per-gene loop even when `n_perms=0` — the gene loop itself is the slow part, not just the permutations. Worth setting even on the fast path.
- **GPU mode** (`ab.use_gpu(True)`) uses a completely different code path and ignores `n_jobs` — not the same code running faster, an actually different implementation.
